In [1]:
import time
import ollama

MODEL = "qwen3:8b"


def call(prompt: str, **options) -> str:
    """封装 ollama.chat，返回回复文本。"""
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options=options,
    )
    return response["message"]["content"]


def header(title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


def run_comparison(prompt: str, param_name: str, values: list, fixed_options: dict = None, runs: int = 2):
    """通用对比函数：对同一 prompt，变化某个参数，每个值跑 runs 次。"""
    fixed = fixed_options or {}
    for val in values:
        print(f"\n  {param_name}={val}:")
        opts = {**fixed, param_name: val}
        for r in range(runs):
            result = call(prompt, **opts)
            # 只取第一行，避免输出太长
            first_line = result.strip().split("\n")[0]
            print(f"    第{r+1}次: {first_line}")

In [2]:
# ============================================================
# 实验 1: Temperature
# ============================================================
def experiment_temperature():
    """
    Temperature 控制输出的随机性。
    - 低温 (0.1): 几乎确定性输出，两次结果基本相同
    - 中温 (0.7): 自然多样，适合对话
    - 高温 (1.5): 高度随机，可能出现意想不到的表达
    """
    header("实验 1: Temperature（温度）")

    prompt = "/no_think 用一句话描述春天"
    print(f"  Prompt: {prompt}")
    print(f"  固定: num_ctx=4096")
    print(f"  观察: 温度越低，两次输出越相似")

    run_comparison(prompt, "temperature", [0.1, 0.5, 0.7, 1.0, 1.5],
                   fixed_options={"num_ctx": 4096})

    # 补充：temperature=0 的确定性
    print(f"\n  --- temperature=0（完全确定性，3次应完全相同）---")
    for r in range(3):
        result = call(prompt, temperature=0, num_ctx=4096)
        first_line = result.strip().split("\n")[0]
        print(f"    第{r+1}次: {first_line}")

In [3]:
experiment_temperature()


  实验 1: Temperature（温度）
  Prompt: /no_think 用一句话描述春天
  固定: num_ctx=4096
  观察: 温度越低，两次输出越相似

  temperature=0.1:
    第1次: 春天是万物复苏的序章，嫩绿的枝芽轻抚大地，绽放的花朵唤醒沉睡的万物，在暖风中谱写生命的诗行。
    第2次: 春风轻拂，万物复苏，繁花似锦，绿意盎然。

  temperature=0.5:
    第1次: 春风轻拂，万物复苏，花开满园，绿意盎然。
    第2次: 春风轻拂，万物复苏，繁花似锦，生机盎然。

  temperature=0.7:
    第1次: 春日暖阳轻抚大地，万物复苏生机盎然。
    第2次: 春风轻拂，万物复苏，花香与鸟鸣交织成生机盎然的画卷。

  temperature=1.0:
    第1次: 春风轻拂，万物复苏，繁花似锦，生机盎然。
    第2次: 春风轻拂，绿意盎然，花香鸟语间万物焕发生机。

  temperature=1.5:
    第1次: 春日暖阳洒落，花绽绿意，万物苏醒，生机盎然。
    第2次: 春风轻拂，万物复苏，花香弥漫，春意盎然。

  --- temperature=0（完全确定性，3次应完全相同）---
    第1次: 春风轻拂，万物苏醒，花香四溢，绿意盎然，人们踏青赏景，欢声笑语中迎接新生的希望。
    第2次: 春风轻拂，万物苏醒，花香四溢，绿意盎然，人们踏青赏景，欢声笑语中迎接新生的希望。
    第3次: 春风轻拂，万物苏醒，花香四溢，绿意盎然，人们踏青赏景，欢声笑语中迎接新生的希望。
